In [6]:
import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [13]:
df = pd.read_excel("Online retail.xlsx", header=None)

df.head()

,0
0,"shrimp,almonds,avocado,vegetables mix,green gr..."
1,"burgers,meatballs,eggs"
2,chutney
3,"turkey,avocado"
4,"mineral water,milk,energy bar,whole wheat rice..."


In [14]:
df.shape

(7501, 1)

In [15]:
df.isnull().sum()

0    0
dtype: int64

## Convert Transactions into List Format

In [16]:
transactions = []

for i in range(len(df)):
    transactions.append(str(df.values[i,0]).split(','))

## Apply Transaction Encoder

In [17]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket = pd.DataFrame(te_array, columns=te.columns_)
basket.head()

,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,True,True,False,True,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False


## Generate Frequent Itemsets

In [18]:
frequent_items = apriori(basket,
                         min_support=0.01,
                         use_colnames=True)

frequent_items.head()

,support,itemsets
0,0.020397,frozenset({almonds})
1,0.033329,frozenset({avocado})
2,0.010799,frozenset({barbecue sauce})
3,0.014265,frozenset({black tea})
4,0.011465,frozenset({body spray})


## Generate Association Rules

In [19]:
rules = association_rules(frequent_items,
                          metric="lift",
                          min_threshold=1)

rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({mineral water}),frozenset({avocado}),0.238368,0.033329,0.011598,0.048658,1.459926,1.0,0.003654,1.016113,0.413630,0.044593,0.015857,0.198329
1,frozenset({avocado}),frozenset({mineral water}),0.033329,0.238368,0.011598,0.348000,1.459926,1.0,0.003654,1.168147,0.325896,0.044593,0.143943,0.198329
2,frozenset({cake}),frozenset({burgers}),0.081056,0.087188,0.011465,0.141447,1.622319,1.0,0.004398,1.063198,0.417434,0.073129,0.059442,0.136473
3,frozenset({burgers}),frozenset({cake}),0.087188,0.081056,0.011465,0.131498,1.622319,1.0,0.004398,1.058080,0.420238,0.073129,0.054892,0.136473
4,frozenset({chocolate}),frozenset({burgers}),0.163845,0.087188,0.017064,0.104150,1.194537,1.0,0.002779,1.018933,0.194767,0.072934,0.018581,0.149934


## View Important Metrics

In [20]:
rules[['antecedents',
       'consequents',
       'support',
       'confidence',
       'lift']].head(10)

,antecedents,consequents,support,confidence,lift
0,frozenset({mineral water}),frozenset({avocado}),0.011598,0.048658,1.459926
1,frozenset({avocado}),frozenset({mineral water}),0.011598,0.348000,1.459926
2,frozenset({cake}),frozenset({burgers}),0.011465,0.141447,1.622319
3,frozenset({burgers}),frozenset({cake}),0.011465,0.131498,1.622319
4,frozenset({chocolate}),frozenset({burgers}),0.017064,0.104150,1.194537
5,frozenset({burgers}),frozenset({chocolate}),0.017064,0.195719,1.194537
6,frozenset({eggs}),frozenset({burgers}),0.028796,0.160237,1.837830
7,frozenset({burgers}),frozenset({eggs}),0.028796,0.330275,1.837830
8,frozenset({french fries}),frozenset({burgers}),0.021997,0.128705,1.476173
9,frozenset({burgers}),frozenset({french fries}),0.021997,0.252294,1.476173


## Filter Strong Rules

In [21]:
strong_rules = rules[
    (rules['confidence'] > 0.3) &
    (rules['lift'] > 1)
]

strong_rules[['antecedents',
              'consequents',
              'support',
              'confidence',
              'lift']]

,antecedents,consequents,support,confidence,lift
1,frozenset({avocado}),frozenset({mineral water}),0.011598,0.348000,1.459926
7,frozenset({burgers}),frozenset({eggs}),0.028796,0.330275,1.837830
38,frozenset({cake}),frozenset({mineral water}),0.027463,0.338816,1.421397
44,frozenset({cereals}),frozenset({mineral water}),0.010265,0.398964,1.673729
58,frozenset({chicken}),frozenset({mineral water}),0.022797,0.380000,1.594172
...,...,...,...,...,...
389,"frozenset({milk, mineral water})",frozenset({spaghetti}),0.015731,0.327778,1.882589
394,"frozenset({spaghetti, olive oil})",frozenset({mineral water}),0.010265,0.447674,1.878079
395,"frozenset({olive oil, mineral water})",frozenset({spaghetti}),0.010265,0.371981,2.136468
401,"frozenset({spaghetti, pancakes})",frozenset({mineral water}),0.011465,0.455026,1.908923


## Sort Rules by Lift

In [22]:
strong_rules = strong_rules.sort_values(by='lift',
                                        ascending=False)

strong_rules.head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
215,frozenset({herb & pepper}),frozenset({ground beef}),0.049460,0.098254,0.015998,0.323450,3.291994,1.0,0.011138,1.332860,0.732460,0.121457,0.249734,0.243136
383,"frozenset({ground beef, mineral water})",frozenset({spaghetti}),0.040928,0.174110,0.017064,0.416938,2.394681,1.0,0.009938,1.416470,0.607262,0.086195,0.294020,0.257474
366,"frozenset({mineral water, frozen vegetables})",frozenset({milk}),0.035729,0.129583,0.011065,0.309701,2.389991,1.0,0.006435,1.260929,0.603138,0.071737,0.206934,0.197546
252,frozenset({soup}),frozenset({milk}),0.050527,0.129583,0.015198,0.300792,2.321232,1.0,0.008651,1.244861,0.599484,0.092158,0.196697,0.209038
226,frozenset({ground beef}),frozenset({spaghetti}),0.098254,0.174110,0.039195,0.398915,2.291162,1.0,0.022088,1.373997,0.624943,0.168096,0.272197,0.312015
395,"frozenset({olive oil, mineral water})",frozenset({spaghetti}),0.027596,0.174110,0.010265,0.371981,2.136468,1.0,0.005460,1.315071,0.547034,0.053621,0.239585,0.215470
343,"frozenset({eggs, ground beef})",frozenset({mineral water}),0.019997,0.238368,0.010132,0.506667,2.125563,1.0,0.005365,1.543848,0.540342,0.040816,0.352268,0.274586
376,"frozenset({ground beef, milk})",frozenset({mineral water}),0.021997,0.238368,0.011065,0.503030,2.110308,1.0,0.005822,1.532552,0.537969,0.044385,0.347493,0.274725
290,frozenset({red wine}),frozenset({spaghetti}),0.028130,0.174110,0.010265,0.364929,2.095966,1.0,0.005368,1.300468,0.538028,0.053472,0.231046,0.211944
284,frozenset({olive oil}),frozenset({spaghetti}),0.065858,0.174110,0.022930,0.348178,1.999758,1.0,0.011464,1.267048,0.535186,0.105651,0.210764,0.239939


## Interpretation & Insights



######  1. Products with high lift are frequently purchased together.
######  2. High confidence rules indicate strong purchasing relationships.
######  3. Market basket analysis helps identify customer buying patterns.
######  4. Frequently associated items can be used for product recommendation and store placement.

# Interview Questions:

#### 1.	What is lift and why is it important in Association rules?

######  Lift measures how strongly two items are associated with each other.

##### Formula:
###### Lift = Confidence / Support of consequent

##### Interpretation:
###### Lift > 1  -> positive association
###### Lift = 1  -> no association
###### Lift < 1  -> negative association

##### Importance:
###### It helps identify meaningful product relationships beyond random chance.

#### 2.	What is support and Confidence. How do you calculate them?

###### Support measures how frequently an itemset appears in the dataset, while Confidence measures the probability that item B is purchased when item A is purchased.
###### Support(A → B) = Transactions containing both A and B / Total Transactions
###### Confidence(A → B) = Transactions containing both A and B / Transactions containing A

#### 3.	What are some limitations or challenges of Association rules mining?

###### Association rule mining can generate a very large number of rules, making it difficult to identify meaningful patterns. It may also produce redundant or irrelevant rules, requires high computational power for large datasets, and does not always capture causal relationships between items. Additionally, selecting appropriate support and confidence thresholds can be challenging.